# Frequentist vs. Bayesian

A frequentist and a Bayesian walk into a bar. They find a coin, toss it 10 times, and see 8 heads. The frequentist says "easy — the probability of heads is 0.8." The Bayesian reflects: *coins are almost always fair*, so they'll adjust their belief, but not by much — "maybe 0.52." The difference is the **prior**: the belief held before gathering data, which frequentists don't use and Bayesians update. This notebook lays out the two philosophies — probability as long-run frequency vs. degree of belief — and simulates both statisticians. It is based on the eighth video of Lesson 6 (Point Estimation).

## 1. Introduction

Everything so far in this lesson — MLE for coins, Gaussians, regression — has been **frequentist**: only the evidence counts. The popcorn-contest twist hinted that scenario plausibility should matter too; this video names the two camps behind that tension, an age-old debate rooted in *what probability even means*.

After completing this notebook you will be able to:

- Tell the bar story with numbers: why 0.8 and why ≈ 0.52 from the same 10 flips.
- State the two interpretations of probability (long-run frequency vs. degree of belief) and the role of the **prior**.
- Contrast the two goals: maximize likelihood vs. update prior beliefs with data.
- Simulate both statisticians and watch them converge as evidence accumulates.

## 2. Theory

### The bar story

Coin found in the street; 10 tosses; **8 heads, 2 tails**.

- **The frequentist:** "Easy — $P(\text{heads}) = 0.8$." Pure evidence: the MLE $\hat p = k/n$.
- **The Bayesian:** "From experience, coins are almost always fair. I have a strong belief $p \approx 0.5$; seeing this evidence I'll adjust — but not by much. Say $0.52$." Still thinks the coin should be fair, now with a little doubt.

The Bayesian introduced a **previous belief — the prior** — representing what they believed *before* the samples, and updated it with the evidence, whether or not the evidence agreed with it.

### Two meanings of probability

| | frequentist | Bayesian |
|---|---|---|
| probability is… | the **long-term frequency** of an event over infinitely many repetitions | the **degree of belief** or certainty in an event |
| prior beliefs | none — only the presented evidence | central — the prior encodes belief before data |
| the goal | find the model with the **greatest likelihood** of generating the observed data | **update the prior** belief about the model using the observations |

### Where this course has been, and where it goes

All the point estimation so far — MLE included — follows the frequentist approach. If you thought a coin was balanced but 10 tosses gave 8 heads, the Bayesian program says: reconsider your assumption *by the right amount*. Next up: digging into the Bayesian approach, and in particular the effect priors have on estimates.

(Worth knowing: neither camp is "wrong." Frequentist methods shine with plentiful data and no trustworthy prior; Bayesian methods shine with scarce data and genuine prior knowledge — and with enough data the two agree, as the simulations below show.)

## 3. Worked Examples

### Example 1: where 0.52 can come from

Encode "coins are almost always fair" as a strong prior centered at 0.5 (e.g. a Beta-like density $\propto p^{a}(1-p)^{a}$ with $a = 100$). Updating on 8H2T gives a posterior peaking at $\frac{100 + 8}{200 + 10} \approx 0.514$ — the Bayesian's "0.5… 2-ish": belief moved toward the evidence, but only slightly, because the prior was strong.

### Example 2: a weak-prior Bayesian

With a mild prior ($a = 2$: "probably fair-ish"), the same data yields $\frac{2 + 8}{4 + 10} \approx 0.71$ — much closer to the frequentist's 0.8. The prior's *strength* dials how much evidence is needed to move you.

### Example 3: the long run reconciles them

After 1,000 tosses of a genuinely 0.8-heads coin (≈800 heads), the strong-prior Bayesian sits at $\frac{100+800}{200+1000} = 0.75$ and climbing; by 10,000 tosses both camps say 0.80 to two decimals. Priors are initial conditions, not destinies.

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(42)

heads, n = 8, 10
p = np.linspace(0.001, 0.999, 999)

def belief_update(a, heads, tails):
    # prior ~ p^a (1-p)^a  updated by the flips -> posterior curve and its peak
    posterior = p**(a + heads) * (1 - p)**(a + tails)
    return posterior / np.trapezoid(posterior, p), (a + heads) / (2 * a + heads + tails)

### The two statisticians answer

In [ ]:
frequentist = heads / n
_, bayesian_strong = belief_update(100, heads, n - heads)
_, bayesian_weak = belief_update(2, heads, n - heads)

print(f"data: {heads} heads in {n} tosses")
print(f"frequentist (MLE):              p = {frequentist:.3f}")
print(f"Bayesian, strong fair prior:    p = {bayesian_strong:.3f}   (the video's ~0.52)")
print(f"Bayesian, weak fair-ish prior:  p = {bayesian_weak:.3f}")

### Belief before and after: the update in pictures

In [ ]:
prior_curve = p**100 * (1 - p)**100
prior_curve /= np.trapezoid(prior_curve, p)
posterior_curve, peak = belief_update(100, heads, n - heads)
likelihood = p**heads * (1 - p)**(n - heads)
likelihood /= np.trapezoid(likelihood, p)

plt.figure(figsize=(9.5, 4.8))
plt.plot(p, prior_curve, color="tab:gray", linewidth=2, label="prior belief (coins are fair)")
plt.plot(p, likelihood, color="tab:blue", linewidth=2, label="likelihood of 8H2T (evidence)")
plt.plot(p, posterior_curve, color="tab:green", linewidth=2, label="updated belief")
plt.axvline(0.8, color="tab:blue", linestyle=":", linewidth=1)
plt.axvline(peak, color="tab:green", linestyle=":", linewidth=1)
plt.title(f"The Bayesian moves from 0.50 toward the evidence -- but only to {peak:.3f}")
plt.xlabel("p (probability of heads)")
plt.ylabel("scaled density")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

### Evidence accumulates: the two camps converge

Toss a true-0.8 coin on and on. The frequentist tracks $k/n$ from the start; the strong-prior Bayesian starts glued to 0.5 and is dragged, toss by toss, to the truth.

In [ ]:
rng = np.random.default_rng(0)

true_p = 0.8
flips = rng.random(10_000) < true_p
ks = np.cumsum(flips)
ns = np.arange(1, 10_001)

freq_est = ks / ns
bayes_est = (100 + ks) / (200 + ns)

for n_check in [10, 100, 1000, 10_000]:
    print(f"after {n_check:>6} tosses: frequentist {freq_est[n_check-1]:.3f}, "
          f"strong-prior Bayesian {bayes_est[n_check-1]:.3f}")

plt.figure(figsize=(9.5, 4.5))
plt.plot(ns, freq_est, color="tab:blue", label="frequentist  k/n")
plt.plot(ns, bayes_est, color="tab:green", label="Bayesian (strong fair prior)")
plt.axhline(true_p, color="black", linestyle="--", linewidth=1, label="true p = 0.8")
plt.xscale("log")
plt.xlabel("number of tosses (log)")
plt.ylabel("estimate of p")
plt.title("Priors are initial conditions, not destinies: both camps reach the truth")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

### Who is better at small n? A tournament

Draw random *fair-ish* coins (the world the Bayesian believes in), give each statistician 10 tosses, and score their squared errors — then repeat in a world of wild coins where the prior is wrong.

In [ ]:
rng = np.random.default_rng(1)

def tournament(coin_sampler, trials=100_000, n_flips=10):
    ps = coin_sampler(trials)
    ks = rng.binomial(n_flips, ps)
    freq = ks / n_flips
    bayes = (100 + ks) / (200 + n_flips)
    return np.mean((freq - ps)**2), np.mean((bayes - ps)**2)

fair_world = lambda t: np.clip(rng.normal(0.5, 0.03, t), 0.01, 0.99)
wild_world = lambda t: rng.uniform(0.05, 0.95, t)

for name, world in [("fair-ish coins ", fair_world), ("wild coins     ", wild_world)]:
    mse_f, mse_b = tournament(world)
    print(f"{name}: MSE frequentist {mse_f:.5f}   MSE Bayesian {mse_b:.5f}"
          f"   -> {'Bayesian' if mse_b < mse_f else 'frequentist'} wins")
print("\nA good prior is a superpower at small n; a wrong prior is a handicap.")

## 5. Visualizations

### The bar story as one picture

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4), sharey=True)

likelihood_curve = p**8 * (1 - p)**2
likelihood_curve /= np.trapezoid(likelihood_curve, p)
axes[0].plot(p, likelihood_curve, color="tab:blue", linewidth=2)
axes[0].axvline(0.8, color="tab:blue", linestyle="--", label="answer: 0.8")
axes[0].set_title('the frequentist: "easy, 8/10 = 0.8"\n(evidence only)')

axes[1].plot(p, prior_curve, color="tab:gray", linewidth=1.5, label="prior")
axes[1].plot(p, posterior_curve, color="tab:green", linewidth=2, label="updated belief")
axes[1].axvline(peak, color="tab:green", linestyle="--", label=f"answer: {peak:.2f}")
axes[1].set_title('the Bayesian: "coins are usually fair... 0.51"\n(prior + evidence)')

for ax in axes:
    ax.set_xlabel("p")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)
axes[0].set_ylabel("scaled density")
plt.suptitle("Same 10 flips, two philosophies")
plt.tight_layout()
plt.show()

### The prior-strength dial

One data set (8H2T), a family of priors from indifferent to adamant: the answer slides from the frequentist's 0.8 to the immovable 0.5.

In [ ]:
strengths = [0, 1, 2, 5, 10, 30, 100, 300]
answers = [(a + 8) / (2 * a + 10) for a in strengths]

plt.figure(figsize=(9, 4.5))
plt.plot(strengths, answers, "o-", color="tab:purple")
plt.axhline(0.8, color="tab:blue", linestyle="--", label="frequentist 0.8")
plt.axhline(0.5, color="tab:gray", linestyle=":", label="prior center 0.5")
plt.xscale("symlog", linthresh=1)
plt.xlabel("prior strength a  (0 = no prior)")
plt.ylabel("estimate after 8H2T")
plt.title("From frequentist to stubborn: the prior-strength spectrum")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

### The whole story on one page

Everything above in a single sheet: the bar story, the prior-strength dial, the two camps converging over 10,000 tosses, and the small-sample tournament where a right prior helps and a wrong one hurts.

<img src="img/07_frequentist_vs_bayesian.png" alt="Concept sheet for this notebook" width="900">

## 6. Exercises

### Basic

**Exercise 1.** Rerun the bar story with 6 heads in 10 tosses. What does each statistician answer (strong prior $a = 100$)? Why is the disagreement smaller than with 8 heads?

**Exercise 2.** Classify each statement as frequentist or Bayesian: (a) "this die lands 6 one-sixth of the time in the long run"; (b) "I'm 90% sure it will rain tomorrow"; (c) "the MLE for the regression line"; (d) "given what we know about drug chemistry, effect sizes this large are implausible."

### Intermediate

**Exercise 3.** The Bayesian's answer $\frac{a + k}{2a + n}$ is a *weighted average* of the prior center 0.5 and the frequentist's $k/n$. Derive the weights explicitly, and interpret $2a$ as "phantom flips."

**Exercise 4.** How much evidence to overcome the prior? For the strong prior ($a = 100$) and a true-0.8 coin, find (by the formula or the simulation) how many tosses are needed before the Bayesian's estimate exceeds 0.75.

### Advanced

**Exercise 5.** Design an experiment where the frequentist embarrasses themselves: a coin tossed only *twice*, both heads. What do the two camps report, and which report would you act on? Connect to the MLE edge cases from the Bernoulli notebook.

**Exercise 6.** Priors from data ("empirical Bayes"): suppose you've previously measured 1,000 street coins, finding heads-rates tightly clustered around 0.5 (std 0.02). Justify a prior strength $a$ from that history (match the prior's spread $\approx \frac{0.5}{\sqrt{2a+1}}$... or numerically), and rerun the bar story with your data-derived prior. Is the resulting answer defensible to a frequentist?

## 7. Solutions

In [ ]:
# Exercise 1: 6 heads in 10
k6 = 6
freq6 = k6 / 10
bayes6 = (100 + k6) / (200 + 10)
print(f"Exercise 1: frequentist {freq6:.2f}, Bayesian {bayes6:.3f}")
print("With 6H the evidence (0.6) sits close to the prior (0.5), so there is")
print("little to disagree about -- disagreement grows with distance between")
print("prior and evidence, and 8H was farther out.")

In [ ]:
# Exercise 2: classifying statements
print("Exercise 2:")
print("(a) frequentist -- long-run frequency over repetitions.")
print("(b) Bayesian -- degree of belief about a one-off event.")
print("(c) frequentist -- likelihood-only estimation.")
print("(d) Bayesian -- prior knowledge shaping what's plausible.")

In [ ]:
# Exercise 3: the weighted-average form
# (a + k)/(2a + n) = (2a/(2a+n)) * 1/2 + (n/(2a+n)) * (k/n)
a, k_, n_ = 100, 8, 10
w_prior = 2 * a / (2 * a + n_)
w_data = n_ / (2 * a + n_)
print(f"Exercise 3: weights: prior {w_prior:.3f}, data {w_data:.3f}")
print(f"check: {w_prior * 0.5 + w_data * (k_ / n_):.4f} = {(a + k_) / (2 * a + n_):.4f}")
print("The prior acts like 2a phantom flips that came up half heads: a = 100")
print("means 'I have already seen 200 fair flips' -- which is why 10 real ones")
print("barely move the needle.")

In [ ]:
# Exercise 4: tosses needed to reach 0.75
# (100 + 0.8 n)/(200 + n) = 0.75  =>  100 + 0.8 n = 150 + 0.75 n  =>  n = 1000
n_star = 50 / 0.05
print(f"Exercise 4: (100 + 0.8n)/(200 + n) = 0.75 -> n = {n_star:.0f} tosses")
print("Matches the convergence plot: after ~1000 tosses the strong-prior")
print("Bayesian passes 0.75 on the way to 0.8.")

In [ ]:
# Exercise 5: two tosses, two heads
print("Exercise 5: frequentist: p = 2/2 = 1.0 -- 'this coin CANNOT land tails',")
print(f"Bayesian (a=100): (100+2)/(200+2) = {(102/202):.3f} -- 'almost surely fair'.")
print("Acting on p = 1 after two flips is absurd (bet anything on heads?);")
print("the Bayesian's near-0.5 is the sane report. This is the Bernoulli")
print("notebook's boundary overfit, cured by exactly the mechanism -- a prior --")
print("that MAP formalizes.")

In [ ]:
# Exercise 6: an empirical prior
rng = np.random.default_rng(3)

# choose a so the prior std over p matches the observed 0.02
# for prior ~ p^a(1-p)^a: std ~ 1/(2 sqrt(2a + 3)); solve 0.02 = ...
a_emp = int(((1 / 0.02)**2 / 4 - 3) / 2)
_, ans = belief_update(a_emp, 8, 2)
print(f"Exercise 6: matching std 0.02 gives a ~ {a_emp}")
print(f"bar-story answer with the empirical prior: {ans:.3f}")
print("Defensible even to a frequentist: this prior IS long-run frequency data")
print("-- 1,000 measured coins -- not mere opinion. Empirical Bayes blurs the")
print("camps: yesterday's evidence becomes today's prior.")

## 8. Conclusion

- The bar story splits statistics: from 8H2T the **frequentist** reports the likelihood-only $\hat p = 0.8$; the **Bayesian**, holding a strong prior that street coins are fair, updates only slightly — ≈ 0.52.
- The root difference is the meaning of probability: **long-run frequency** (frequentist) vs. **degree of belief** (Bayesian) — and with it, the legitimacy of the **prior**, the belief held before data.
- Two goals: maximize $P(\text{data} \mid \text{model})$ vs. update $P(\text{model})$ into a belief that accounts for the observations.
- The quantitative bridge: the Bayesian answer is a weighted average of prior center and data ($2a$ "phantom flips"), so priors dominate at small $n$, help enormously when they're right, hurt when wrong — and *wash out* as evidence accumulates, reconciling the camps.
- Everything so far (MLE included) was frequentist; next the course goes Bayesian in earnest — priors, posteriors, and **MAP**.